# Importancia de variables del champion (#16)

Champion elegido en #22: **XGBoost, prueba Optuna 34**. Se calcula con train únicamente, usando permutación sobre los cinco folds de validación y el Pipeline completo. No se evalúa test.


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import subprocess
import sys

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.preprocessing import load_train_test, RAW_DATA_PATH
from src.xgboost_model import FIXED_PARAMS, encode_target, make_folds, make_pipeline
from src.feature_importance import (
    permutation_feature_importance,
    validate_champion_candidate,
    validate_effective_params,
)

# Datos y procedencia del champion.
X_train, reserved_X, y_labels, reserved_y, thresholds = load_train_test(return_thresholds=True)
del reserved_X, reserved_y
y_train = encode_target(y_labels)
folds = make_folds(X_train, y_train)

details = json.loads((ROOT / "docs/results/xgboost_details.json").read_text(encoding="utf-8"))
comparison = pd.read_csv(ROOT / "docs/results/xgboost_comparison.csv")
raw_sha256 = hashlib.sha256(RAW_DATA_PATH.read_bytes()).hexdigest()

selected_trial, selected_params, candidate_params = validate_champion_candidate(
    details, comparison, X_train, thresholds, raw_sha256, fixed_params=FIXED_PARAMS
)
pipeline = make_pipeline(selected_params)
validate_effective_params(pipeline, candidate_params)
print(f"Validacion del champion OK: XGBoost, prueba Optuna {selected_trial}")

code_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
provenance = {
    "snapshot": details["provenance"]["snapshot"],
    "raw_sha256": raw_sha256,
    "code_commit": code_commit,
}

importance = permutation_feature_importance(
    pipeline,
    X_train,
    y_train,
    folds,
    model_name=f"XGBoost champion (trial {selected_trial})",
    n_repeats=10,
    provenance=provenance,
)

output = ROOT / "docs/results/champion_feature_importance.csv"
with output.open("w", encoding="utf-8") as file:
    file.write(f"# Feature importance del champion: XGBoost (prueba Optuna {selected_trial})\n")
importance.to_csv(output, mode="a", index=False)

expected_params = {**FIXED_PARAMS, **selected_params}
versions_source = details["provenance"].get("versions", {})
versions_installed = {
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scipy": __import__("scipy").__version__,
    "scikit-learn": __import__("sklearn").__version__,
    "xgboost": __import__("xgboost").__version__,
    "optuna": __import__("optuna").__version__,
    "threadpoolctl": __import__("threadpoolctl").__version__,
}
metadata = {
    "champion": "XGBoost",
    "selected_trial": selected_trial,
    "selected_params": selected_params,
    "candidate_csv_params_match": candidate_params == expected_params,
    "effective_params_match": True,
    "train_rows": len(X_train),
    "thresholds": thresholds,
    "raw_sha256": raw_sha256,
    "code_commit": code_commit,
    "python": platform.python_version(),
    "versions_source": versions_source,
    "versions_installed": versions_installed,
}
(ROOT / "docs/results/champion_feature_importance.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

display(importance[["feature", "importance_mean", "importance_std"]])
print(f"Guardado: {output}")


Validacion del champion OK: XGBoost, prueba Optuna 34


,feature,importance_mean,importance_std
0,minimum_nights,0.200013,0.005691
1,accommodates,0.043586,0.005154
2,latitude,0.029682,0.005644
3,bathrooms_count,0.018728,0.001533
4,longitude,0.015481,0.004052
5,property_type,0.008967,0.003366
6,beds,0.007010,0.003604
7,is_shared_bathroom,0.006657,0.002497
8,has_dishwasher,0.005813,0.002113
9,room_type,0.004524,0.000722


Guardado: C:\Users\mmate\OneDrive\Documents\Bootcamp F5\Modulo 2\Proyecto 6\bcn-price-classifier\docs\results\champion_feature_importance.csv
